# 13. OpenAI Agents SDK — 규정 도우미 완성

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/14_agents_sdk.ipynb)

> 🧭 **스토리** — 13에서 손으로 짠 툴 루프가 돌아간다. 이제 반복문은 SDK에 맡기고, 실제로 내놓을 수 있게 **안전장치**(턴 제한·출력 검사)와 **분업**(핸드오프)을 붙인다. 하루의 마지막 조각이다.
>
> 📖 **이론 연결** — 이론2 p75~81 (스키마 설계·5패턴·도구 작성 원칙·Agents SDK 실행 통제·Guardrail·HITL·Tracing·안전장치 위치) · ⏱ **11분** · 🖥 **CPU**
>
> GPU가 필요 없습니다. 기본 런타임(CPU)으로 실행하세요.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
`openai-agents`는 OpenAI가 공개한 에이전트 SDK입니다(설치 이름 `openai-agents`, import 이름 `agents`). 검색 도구는 12와 같은 BM25를 씁니다. GPU는 필요 없습니다.

In [ ]:
%pip -q install openai-agents rank_bm25 kiwipiepy

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-data/genai)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-data/main/genai/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.
print("키 입력 완료 · 모델:", MODEL_API)

In [ ]:
# ⚙️ API 오류를 한국어로 설명하는 헬퍼 (노트북 12과 같음)
import openai
def explain_error(e):
    if isinstance(e, openai.AuthenticationError):
        print("🔑 401 인증 실패 — 키가 없거나 틀렸습니다. 키 셀을 다시 실행하세요.")
    elif isinstance(e, openai.RateLimitError):
        print("💸 429 지출 한도 초과(project_spend_limit_exceeded) — 강사에게 알려 주세요." if "project_spend_limit_exceeded" in str(e)
              else "⏳ 429 속도 제한 — 잠시 뒤 다시 실행하세요.")
    else:
        print("⚠️", type(e).__name__, str(e)[:200])

### 트레이싱 켜고 끄기 ⚙️
Agents SDK는 실행 기록(**트레이스**)을 기본으로 OpenAI 대시보드(https://platform.openai.com/traces)에 보냅니다. 트레이스는 **키 주인의 조직**에 쌓입니다. 오늘은 전원이 같은 수업용 키를 쓰므로 **모든 수강생의 트레이스가 강사 대시보드에 모입니다.** 강사가 화면으로 보여 줄 예정입니다. 질문에 개인정보를 넣지 마세요. 보내고 싶지 않으면 `True`로 바꾸고 실행하세요.

In [ ]:
from agents import Agent, Runner, function_tool, set_tracing_disabled, ModelSettings
from openai.types.shared import Reasoning
DISABLE_TRACING = False          # ← True로 바꾸면 트레이스를 보내지 않습니다
if DISABLE_TRACING:
    set_tracing_disabled(True)
print("트레이싱:", "끔" if DISABLE_TRACING else "켬 (강사 대시보드로 전송)")

## 1. `@function_tool` — 함수가 곧 도구 🔵
12에서는 JSON 스키마를 손으로 썼습니다. SDK는 **타입 힌트와 독스트링**을 읽어 스키마를 만들어 줍니다. 그래서 독스트링이 곧 모델이 읽는 설명서입니다.

In [ ]:
import ast, operator, json, datetime
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

CHUNKS = load_json("hanbit_chunks_gt.json")
kiwi = Kiwi()
def ko_tokens(text):
    return [t.form for t in kiwi.tokenize(text) if t.tag[0] in "NVSXM" and t.tag not in ("SF", "SP", "SS", "SE", "SO", "SW")]
bm25 = BM25Okapi([ko_tokens(c["text"]) for c in CHUNKS])

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg}
def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS: return OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPS: return OPS[type(node.op)](_eval(node.operand))
    raise ValueError("허용하지 않는 수식")

@function_tool
def search_rules(query: str) -> str:
    """한빛테크 임직원 출장·복리후생 규정집에서 질문과 관련된 조항·표 3개를 찾는다. 금액·일수·조건 등 규정 사실이 필요하면 먼저 호출한다.

    Args:
        query: 검색어. 예: '과장 국내 출장 일비 숙박비'
    """
    scores = bm25.get_scores(ko_tokens(query))
    top = sorted(range(len(CHUNKS)), key=lambda i: -scores[i])[:3]
    return json.dumps([{"id": CHUNKS[i]["id"], "text": CHUNKS[i]["text"]} for i in top], ensure_ascii=False)

@function_tool
def calc(expression: str) -> str:
    """사칙연산 수식을 계산한다. 금액 합계·곱셈은 암산하지 말고 이 도구를 쓴다.

    Args:
        expression: 파이썬 수식. 예: '30000*3 + 110000*2'
    """
    try:
        return str(_eval(ast.parse(expression.replace(",", ""), mode="eval").body))
    except Exception as e:
        return f"error: {e}"

@function_tool
def today() -> str:
    """오늘 날짜(YYYY-MM-DD)를 돌려준다."""
    return datetime.date.today().isoformat()

for t in [search_rules, calc, today]:
    print(t.name, "→", json.dumps(t.params_json_schema, ensure_ascii=False))

💡 **해석** — `required`와 `additionalProperties: false`가 자동으로 붙었습니다. 13에서 손으로 맞춘 strict 스키마와 같습니다(이론2 p75). 독스트링의 `Args:` 설명은 각 파라미터의 `description`으로 들어갑니다.

## 2. 에이전트 = 모델 + 도구 + 지시 🔵
에이전트의 3요소(이론2 p68)를 `Agent` 하나에 담고 `Runner.run`으로 돌립니다. 13의 `while` 루프를 SDK가 대신 돌립니다.
노트북(코랩)에는 이미 이벤트 루프가 돌고 있으므로 `Runner.run_sync` 대신 **`await Runner.run(...)`**을 씁니다.

In [ ]:
INSTRUCTIONS = ("너는 한빛테크 규정 도우미다. 규정 사실은 search_rules로 찾은 근거로만 답하고, 계산은 calc로 한다. "
                "답의 끝에 근거 청크 id를 [s2_t1_domestic]처럼 대괄호로 적는다. 근거가 없으면 모른다고 답한다.")
SETTINGS = ModelSettings(reasoning=Reasoning(effort="low"))   # 생각은 조금만 (비용·속도)

agent = Agent(name="rules_helper", instructions=INSTRUCTIONS, tools=[search_rules, calc, today],
              model=MODEL_API, model_settings=SETTINGS)
QUESTION = "과장이 부산에 2박 3일 출장 가면 일비와 숙박비 상한 합계는?"
result = None
try:
    result = await Runner.run(agent, QUESTION, max_turns=5)
    print(result.final_output)
except Exception as e:
    explain_error(e)

💡 **해석** — 13에서 50줄로 짠 루프가 `Runner.run` 한 줄이 되었습니다. 기대 답은 **310,000원**과 근거 `[s2_t1_domestic]`입니다. `max_turns=5`는 모델 호출 횟수의 상한입니다. 넘으면 예외가 납니다(4절).

## 3. 무슨 일이 있었나 — `new_items`로 실행 순서 보기 🔵
`result.new_items`에 실행 중 생긴 항목이 순서대로 남습니다. 도구 호출(`tool_call_item`) → 결과(`tool_call_output_item`) → 답(`message_output_item`)을 따라가 봅니다.

In [ ]:
if result is not None:
    for i, it in enumerate(result.new_items, 1):
        raw = it.raw_item
        if it.type == "tool_call_item":
            print(f"{i}. 🛠  호출  {raw.name}({raw.arguments})")
        elif it.type == "tool_call_output_item":
            print(f"{i}. 📦 결과  {str(it.output)[:80]}…")
        elif it.type == "message_output_item":
            print(f"{i}. 💬 답    {result.final_output[:80]}")
        else:
            print(f"{i}. ({it.type})")
    print("토큰 합계:", sum(r.usage.total_tokens for r in result.raw_responses))
else:
    print("2절 실행이 실패해서 건너뜁니다.")

💡 **해석** — `search_rules` → `calc` → 답 순서가 보이면 13의 손 루프와 똑같이 움직인 것입니다. `reasoning_item`은 모델이 도구를 고르기 전에 생각한 흔적입니다. 이 기록이 그대로 **트레이스**가 되어 대시보드에 남습니다. 어디서 틀렸는지 찾을 때 가장 먼저 보는 곳입니다.

## 4. 안전장치 ① — 턴 제한 `max_turns` 🔵
모델이 도구를 끝없이 부르면 비용이 계속 나갑니다. `max_turns=1`로 일부러 막아 봅니다. 이 질문은 최소 2턴(도구 호출 → 답)이 필요합니다.

In [ ]:
from agents import MaxTurnsExceeded
try:
    await Runner.run(agent, QUESTION, max_turns=1)
except MaxTurnsExceeded as e:
    print("🛑 MaxTurnsExceeded:", e)
except Exception as e:
    explain_error(e)

💡 **해석** — 1턴 안에 끝내지 못해 예외로 멈췄습니다. 서비스에서는 이 예외를 잡아 "답을 찾지 못했습니다. 담당 부서에 문의하세요"로 바꿔 보여 줍니다. 비용 상한과 무한 루프 방지를 함께 얻습니다(이론2 p78).

## 5. 안전장치 ② — 출력 가드레일 🔵
최종 답이 나간 **뒤가 아니라 나가기 전에** 검사합니다. 규칙: "근거 id(대괄호)가 없는 답은 내보내지 않는다." 위반하면 `tripwire`가 걸려 예외가 납니다.
일부러 근거를 적지 말라고 지시한 에이전트로 가드레일을 걸리게 해 봅니다.

In [ ]:
from agents import output_guardrail, GuardrailFunctionOutput, OutputGuardrailTripwireTriggered

@output_guardrail
async def cite_check(ctx, agent, output) -> GuardrailFunctionOutput:
    has_cite = "[" in str(output)
    return GuardrailFunctionOutput(output_info={"근거_id_있음": has_cite}, tripwire_triggered=not has_cite)

safe_agent = agent.clone(output_guardrails=[cite_check])          # 근거를 적는 정상 에이전트 + 가드레일
sloppy_agent = agent.clone(instructions="규정 질문에 숫자만 짧게 답한다. 근거 id는 적지 않는다.",
                           output_guardrails=[cite_check])          # 근거를 안 적는 에이전트 + 가드레일
for name, a in [("정상", safe_agent), ("근거 없음", sloppy_agent)]:
    try:
        r = await Runner.run(a, QUESTION, max_turns=5)
        print(f"[{name}] ✅ 통과:", r.final_output[:100])
    except OutputGuardrailTripwireTriggered as e:
        print(f"[{name}] 🚨 가드레일 차단:", e.guardrail_result.output.output_info)
    except Exception as e:
        explain_error(e)

💡 **해석** — 정상 에이전트는 통과하고, 근거를 빼먹은 에이전트는 차단됩니다. 오늘은 대괄호 하나만 봤지만 실무에서는 "근거 id가 실제 검색 결과에 있었나", "개인정보가 들어 있나"를 검사하거나 작은 모델에게 판정을 맡깁니다. **입력 가드레일**(질문이 범위 밖인지 미리 거르기)도 같은 방식으로 붙입니다.

## 6. 분업 — 핸드오프 🔵
접수 에이전트가 질문을 읽고 **규정 담당** 또는 **잡담 담당**에게 넘깁니다. Anthropic 5패턴 중 **라우팅**입니다(이론2 p76). SDK는 넘기기를 `transfer_to_<이름>`이라는 도구로 모델에게 보여 줍니다.

⚠️ 에이전트 `name`은 **영문·숫자·밑줄**로 지으세요. 강사 환경에서 이름을 "규정 도우미"처럼 한글로 지었더니 두 넘기기 도구가 모두 `transfer_to_______`로 바뀌어 이름이 겹쳤고, 규정 질문이 잡담 담당에게 갔습니다. 한글 설명은 `handoff_description`에 적습니다.

In [ ]:
chat_agent = Agent(name="chitchat", handoff_description="인사·잡담 담당",
                   instructions="한빛테크 사내 챗봇으로서 잡담에 한두 문장으로 친절히 답한다.",
                   model=MODEL_API, model_settings=SETTINGS)
rules_agent = agent.clone(handoff_description="출장·휴가·복리후생 등 회사 규정 질문 담당")
triage = Agent(name="triage", instructions="규정 질문이면 rules_helper에게, 잡담이면 chitchat에게 넘겨라. 직접 답하지 않는다.",
               handoffs=[rules_agent, chat_agent], model=MODEL_API, model_settings=SETTINGS)

for q in ["오늘 금요일이라 기분 좋네요!", "부장의 국내 출장 숙박비 상한은?"]:
    try:
        r = await Runner.run(triage, q, max_turns=6)
        print(f"Q: {q}\n  → 담당: {r.last_agent.name} | 순서: {[it.type for it in r.new_items]}\n  → 답: {r.final_output[:100]}\n")
    except Exception as e:
        explain_error(e)

💡 **해석** — `last_agent`가 질문에 따라 `chitchat`과 `rules_helper`로 갈립니다. 순서에 `handoff_call_item` → `handoff_output_item`이 보이면 접수 에이전트가 넘기기 도구를 부른 것입니다. 담당마다 도구·지시를 따로 두면 각자의 지시문이 짧아지고, 잡담에 검색 비용을 쓰지 않습니다.

## 7. 트레이스 대시보드 — 강사 시연 🔵
강사 화면에서 https://platform.openai.com/traces 를 엽니다. 방금 돌린 실행마다 트레이스가 하나씩 있고, 펼치면 **에이전트 → 모델 호출 → 도구 호출 → 가드레일 → 핸드오프**가 시간 막대로 보입니다. 각 단계의 입력·출력·토큰·지연을 클릭해서 봅니다.
- 수업용 공용 키라서 모든 수강생의 실행이 한 대시보드에 섞입니다. 실무에서는 서비스·환경별로 프로젝트와 키를 나눕니다.
- 개인정보·기밀을 다루는 서비스는 트레이스를 끄거나(`set_tracing_disabled`) 민감 데이터 기록을 끄는 설정을 씁니다.

## 8. 바꿔서 실행 🟡
규정에 근거가 없는 질문에는 "모른다"보다 **어디에 물어야 하는지**를 알려 주는 편이 친절합니다.
1. 아래 지시문 그대로 실행해 근거 없는 질문의 답을 봅니다.
2. `MY_INSTRUCTIONS`나 `QUESTIONS`를 바꿔 다시 실행합니다.
가드레일도 함께 바꿨습니다. 근거 id가 있거나 "담당 부서 안내"로 끝나면 통과입니다.

In [ ]:
MY_INSTRUCTIONS = INSTRUCTIONS + (" 규정에서 근거를 찾지 못하면 추측하지 말고 "
                                  "'담당 부서 안내: 인사팀(hr@hanbit-tech.example)'이라는 문장으로 답을 끝낸다.")   # ← 바꿔 보세요

@output_guardrail
async def cite_or_dept(ctx, agent, output) -> GuardrailFunctionOutput:
    ok = "[" in str(output) or "담당 부서 안내" in str(output)
    return GuardrailFunctionOutput(output_info={"통과": ok}, tripwire_triggered=not ok)

my_agent = agent.clone(instructions=MY_INSTRUCTIONS, output_guardrails=[cite_or_dept])
QUESTIONS = ["재택근무는 주 며칠까지 가능한가?", "출장비 정산은 복귀 후 며칠 이내에 해야 하나?"]   # ← 바꿔 보세요
for q in QUESTIONS:
    try:
        r = await Runner.run(my_agent, q, max_turns=5)
        print(f"Q: {q}\nA: {r.final_output}\n")
    except OutputGuardrailTripwireTriggered as e:
        print(f"Q: {q}\n🚨 가드레일 차단: {e.guardrail_result.output.output_info}\n")
    except Exception as e:
        explain_error(e)

💡 **해석** — 재택근무는 규정집에 없으므로 "담당 부서 안내: 인사팀…"으로 끝나야 하고, 정산 기한은 `[s6_text]` 근거와 함께 7일이 나와야 합니다. 노트북 11의 평가 질문 중 **근거 없음** 유형(2문항)에 대한 대응 방식을 지시문과 가드레일 두 겹으로 고정한 것입니다.

### 🤔 생각해보기

오늘 만든 규정 도우미를 사내에 실제로 배포하려면 무엇이 더 필요할까요? 다섯 가지 이상 적어 보세요.

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- **평가 세트와 회귀 테스트**: QA 15문항처럼 정답이 있는 질문을 늘려 모델·프롬프트를 바꿀 때마다 자동 채점한다.
- **문서 최신화 파이프라인**: 규정이 개정되면 파싱 → 청킹 → 색인을 다시 돌린다(노트북 06~09). 개정 이력과 시행일을 메타데이터로 둔다.
- **권한과 개인정보**: 사용자별로 볼 수 있는 문서를 제한하고, 질문·답 로그에서 개인정보를 가린다. 트레이스 보관 정책을 정한다.
- **비용·한도 관리**: 프로젝트별 키, 월 하드 한도, 알림, 캐싱, 작은 모델 우선 라우팅(노트북 12).
- **사람 승인**: 금액 지급·결재처럼 되돌리기 어려운 일을 하는 도구는 실행 전에 사람이 승인한다.
- **입력 가드레일·남용 방지**: 범위 밖 질문, 프롬프트 주입, 과도한 호출을 거른다.
- **관측과 피드백**: 트레이스·지연·실패율 대시보드, 사용자 👍/👎 수집, 실패 사례를 평가 세트에 추가.
- **장애 대비**: API 장애 시 대체 모델(예: `gpt-5-nano`나 사내 오픈 모델)로 전환.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- `@function_tool`은 타입 힌트·독스트링으로 strict 스키마를 만든다. 독스트링이 곧 도구 설명서다.
- `await Runner.run(agent, 질문, max_turns=…)`이 13의 툴 루프를 대신한다. `new_items`로 순서를 본다.
- 안전장치는 겹겹이: `max_turns`(비용·루프), 출력 가드레일(근거 없는 답 차단), 지시문(근거 없으면 담당 부서 안내).
- 핸드오프로 라우팅한다. 에이전트 이름은 영문으로 짓는다.
- 트레이스는 키 주인의 대시보드에 쌓인다. 공용 키·민감 데이터면 끄는 것을 고려한다.

> 다음 노트북: **15_decisions_api — 생성하지 않고 고른다: 라우팅·채점·가드레일을 결정 API로**